In [3]:


import torch
import numpy as np
import nrrd
from   monai.networks.nets import UNet
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, ScaleIntensityRanged, ToTensord
)
from monai.data import DataLoader
import pytorch_lightning as pl

# Define the transformation pipeline you used in training, excluding random augmentations
infer_transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    ScaleIntensityRanged(keys=['image'], a_min=-1100, a_max=3000, b_min=0.0, b_max=1.0, clip=True),
    ToTensord(keys=["image"]),
])
class UNetModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        # self.global_image_counter = 0
        # self.global_image_counter = 0
        self.model = UNet(
            spatial_dims=3,
            in_channels=1,
            out_channels=3,  # For two objects + background
            channels=(16, 32, 64, 128, 256),
            strides=(2, 2, 2, 2),
            num_res_units=2,
            norm='batch',
        )
    
    def forward(self, x):
        x = self.model(x)
        return x

# Load the model structure
model = UNetModel()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Y:\DeepLearningMONAI\jobs\JI-4\model
# Load the trained weights
trained_model_path = 'Y:/DeepLearningMONAI/jobs/model/JI-4/Nate_JI-1_Model.pth'  # Path to your saved model weights
model.load_state_dict(torch.load(trained_model_path, map_location=device))
model.eval()

# Prepare the device for inference (use a GPU if available)

model.to(device)

# Load a new CT scan for inference
# Y:\DeepLearningMONAI\ALLDATASAMEGEOMETRY
ct_scan_path = 'Y:/DeepLearningMONAI/ALLDATASAMEGEOMETRY/Case_04_Vx3.nrrd'  # Specify the path to your new CT scan file
data = {'image': ct_scan_path}
data = infer_transforms(data)  # Apply the transformations
image = data['image'].unsqueeze(0).to(device)  # Add batch dimension and send to device

# Perform inference
with torch.no_grad():
    outputs = model(image)
    pred = torch.argmax(outputs, dim=1)  # For semantic segmentation, select the class with the highest probability
    pred = pred.squeeze(0).cpu().numpy()  # Remove batch dimension and convert to numpy array

# Save the prediction to a .nrrd file
# Y:\DeepLearningMONAI\Inference_Trials
output_pred_path = 'Y:/DeepLearningMONAI/Inference_Trials/Case_04_Inference.nrrd'  # Specify the path to save the prediction
nrrd.write(output_pred_path, pred)  # Save the segmentation map


ModuleNotFoundError: No module named 'pytorch_lightning'

In [2]:
# Load the checkpoint file
import torch
checkpoint = torch.load("./Nate_JI-1_Model.pth", map_location=torch.device('cpu'))

# Print the keys in the checkpoint dictionary
print(checkpoint.keys())


FileNotFoundError: [Errno 2] No such file or directory: './Nate_JI-1_Model.pth'